# Feature Selection in Covid-19 Dataset and Predictive Model to determine the Risk of Death

## 1. Data pre-processing  

### Importing the relevant libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error 
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
%matplotlib inline
sns.set() 

### Charging data

In [ ]:
df=pd.read_csv('Data.csv',index_col=0) #Your data needs to have a .csv format

In [ ]:
df.head(3) #Visualizing the data

In [ ]:
print('The dataset has %s observations and  %s regressors' % (df.shape[0],df.shape[1]))

### Checking data types

In [ ]:
df.dtypes

In [ ]:
#Generating a dataset copy to don't modify the original file
df_mapped=df.copy()

In [ ]:
# Here, it is neccesary to drop the features that we don't need or what we consider not useful.  
df_mapped.drop(['ENTIDAD_UM','ENTIDAD_NAC','ENTIDAD_RES','MUNICIPIO_RES','PAIS_NACIONALIDAD','PAIS_ORIGEN','MIGRANTE','SECTOR','ORIGEN','ID_REGISTRO','FECHA_INGRESO','FECHA_SINTOMAS','FECHA_DEF','NACIONALIDAD','HABLA_LENGUA_INDIG','INTUBADO','UCI'], axis='columns', inplace=True)

In [ ]:
print('The new dataset has %s observations and  %s regressors' %(df_mapped.shape[0],df_mapped.shape[1]))

### Checking missing values 


In [ ]:
df_mapped.isnull().sum().sort_values(ascending=False).head(18)#counting the empty elements

### Getting the dummies features 

In [ ]:
df_w_dummy = pd.get_dummies(df_mapped, columns=['RESULTADO','SEXO','TIPO_PACIENTE','NEUMONIA','EMBARAZO','DIABETES','EPOC','ASMA','INMUSUPR','HIPERTENSION','OTRA_COM','CARDIOVASCULAR','OBESIDAD','RENAL_CRONICA','TABAQUISMO','OTRO_CASO'],drop_first=True) #'drop first=True' indicates the elimination of the first variable for each categorie to avoid multicolinearity problems 


In [ ]:
df_w_dummy.head() #Visualizing the data with dummies variables

In [ ]:
### Checking data types of the new dataset

In [ ]:
df_w_dummy.dtypes

# 2. Defining the target

In [ ]:
### Here we define that the target is the feature called "Mortalidad"

In [ ]:
df_w_dummy.MORTALIDAD.value_counts() #Checking the number of deaths
#1=survivor
#0=death

In [ ]:
df_muertos = df_w_dummy[df_w_dummy['MORTALIDAD'] == 0]
df_vivos = df_w_dummy[df_w_dummy['MORTALIDAD'] == 1]

# 3. Feature selection process

In [ ]:
#Undersampling and shuffling the data in order to have the same proportion in both classes and generate a robust model

In [ ]:
df_vivos_reduced = df_vivos.sample(df_muertos.shape[0],random_state=103)
df_reduced = pd.concat([df_vivos_reduced,df_muertos],axis=0)
df_reduced = df_reduced.sample(frac=1,random_state=103)
df_reduced = df_reduced.astype('float64')
df_reduced.head()

### Checking the shape of the dataset

In [ ]:
df_reduced.shape

### Verifing the sapmles for each class

In [ ]:
df_reduced.MORTALIDAD.value_counts()

In [ ]:
# Establishing the target and the regressors
y=df_reduced['MORTALIDAD']
x=df_reduced.drop('MORTALIDAD',axis=1)

In [ ]:
print(y.shape)
print(x.shape)

### Splitting the dataset in training and test

In [ ]:
X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(x)
y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.values.reshape(-1, 1))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=103)

### Getting the best variables using the RFE

In [ ]:
from sklearn.feature_selection import RFE #recursive feature elimination
from sklearn.linear_model import LogisticRegression
estimator = LogisticRegression(solver='saga', class_weight='balanced')
selector = RFE(estimator, 15, step=1)
selector = selector.fit(X_train, y_train.ravel())

print(selector.support_)
print(selector.ranking_)

In [ ]:
best_variables = selector.support_
df_no_covid = x
df_no_covid.loc[:, best_variables].head()

In [ ]:
best_variables

In [ ]:
print(selector.support_)
print(selector.ranking_)

# 4. Generating the Logistic regression model and evaluating it

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
# Ejercicio

# En esta parte est? diciendo que de los datos tome 2 variables para x que van a predecir a y
X = x[['EDAD','OTRO_CASO_NO ESPECIFICADO','RESULTADO_POSITIVO COVID', 'DIABETES_SI', 'NEUMONIA_SI','OTRO_CASO_SI','RESULTADO_POSITIVO COVID','CARDIOVASCULAR_SI','HIPERTENSION_SI','INMUSUPR_SI','OTRA_COM_SI','RESULTADO_Resultado Pendiente','TABAQUISMO_SI','ASMA_SI','OBESIDAD_SI']]
y = y

# Hacemos un split de los datos en conjunto de entrenamiento y conjunto de prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101)

# Se hace el ajuste y se calculan las variables de salida para entrenamiento y para prueba
clf = LogisticRegression(solver='lbfgs').fit(X_train, y_train)
y_train_hat = clf.predict(X_train)
y_test_hat = clf.predict(X_test)

# Se genera la figura en blanco de 0 a 1
plt.figure(figsize = (10, 6))
plt.plot([0,1], [0,1], 'r--')

# Se genera la grafica ROC y sus probabilidades as? como los datos de la grafica
probs = clf.predict_proba(X_test)
probs = probs[:, 1]
fpr, tpr, thresholds = roc_curve(y_test,probs)
roc_auc = roc_auc_score(y_test, y_test_hat)
label = 'AUC:' + ' {0:.2f}'.format(roc_auc)
plt.plot(fpr, tpr, c = 'g', label = label, linewidth = 4)
plt.xlabel('FPR', fontsize = 16)
plt.ylabel('TPR', fontsize = 16)
plt.title('ROC', fontsize = 16)
plt.legend();
plt.savefig('ROC.pdf')

### Making a classification report

In [ ]:
target_names = ["Class {}".format(i) for i in range(2)]
print(classification_report(y_test, y_test_hat, target_names=target_names))

### Generating a confusion matrix

In [ ]:
labels = [0,1]
confusion_matrix(y_test, y_test_hat,labels = labels)

cm = confusion_matrix(y_test, y_test_hat,labels)
sns.heatmap(cm,
            annot=True,
            fmt='d',
            xticklabels=labels,
            yticklabels=labels,
            cmap='Greens')
plt.ylabel('Predicted diagnostic')
plt.xlabel('Actual diagnostic');
plt.title('Confusion Matrix')
plt.savefig('Confusion matrix.pdf')
